# Spatial Neural Networks

Using a single point of data for predicting SWF is the logical method from a purely phisical context. However, if our aim is to get the most accurate reflection of reality, we may want to include more context to our input, especially geographical context.

In this notebook, we'll explore the benefits of predicting SWF of a point using a window of data centered in said point instead of just the point itself.

### Library loading

In [40]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import time
import optuna
import shap
from tabulate import tabulate
from sklearn.model_selection import train_test_split

import json
from pathlib import Path

from CIMR.SurfaceWaterFraction_ATBD_main.algorithm.processing.validation_data_processing import load_lut, unravel_freqpol, atmospheric_corrections

from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error, make_scorer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import xgboost as xgb

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

### Data setup

We need to create windows for every valid point.

In [41]:
# Window size
WINDOW_SIZE = 5

# Radius
RADIUS = WINDOW_SIZE // 2

In [42]:
# Load the LUT with reference land emissivities
lut_h_path = "CIMR/SurfaceWaterFraction_ATBD_main/data/lut_de_lannoy_K_h.csv"
lut_v_path = "CIMR/SurfaceWaterFraction_ATBD_main/data/lut_de_lannoy_K_v.csv"

local_tds = "CIMR/SurfaceWaterFraction_ATBD_main/data/CIMR_SWF_TDS_JNB_v3.nc"

tds_base = xr.open_dataset(local_tds)

# NOTE: we need to normalize VOD between 0 and 1 from original LPDR's values: [0-3] Neper
min_val = 0
max_val = 3
old_attrs = tds_base.VOD.attrs
tds_base["VOD"] = (tds_base.VOD - min_val) /(max_val - min_val)

old_attrs.update(
    {
        "Valid_range": "1-0",
        "Description": "NORMALIZED VOD, original is 0-3 Neper", "Units": None
    }
)
tds_base.VOD.attrs = old_attrs

tds_procesado = tds_base.copy()

tds_procesado = atmospheric_corrections(tds_procesado)

tds_procesado = load_lut(tds_procesado, lut_filepath=lut_h_path)
tds_procesado = load_lut(tds_procesado, lut_filepath=lut_v_path)

# unravel tbtoa layer into frequency and polarization arrays
tds_procesado = unravel_freqpol(tds_procesado, dvars=[
    "tbtoa", "tbboa_de_lannoy"
])

# Drop the multy dimentional dvars that we dont need anymore
tds_procesado = tds_procesado.drop_vars(["tbtoa", "tbboa_de_lannoy"])

# Get emissivity from TDS, Which has ERA5 skin temperature
for freq in ["19", "37"]:
    for pol in ["H", "V"]:
        tds_procesado[f"emiss{freq}{pol}_de_lannoy"] = tds_procesado[f"tbboa_de_lannoy{freq}{pol}"] / tds_procesado["surtep_ERA5"]

# Use IGBP to remove the ocean
tds_procesado = tds_procesado.where(tds_procesado.IGBP_landcover > 0)

tds_ingenieria = tds_procesado.copy()

# SWF computation
ref_water_emiss_h = 0.288760

tds_ingenieria["Caracteristica1"] = (1) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["Caracteristica2"] = (tds_ingenieria.ref_land_emis_de_lannoy_K_h) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["Caracteristica3"] = (tds_ingenieria.emiss19H_de_lannoy) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["SWF_calculada"] = (tds_ingenieria.ref_land_emis_de_lannoy_K_h - tds_ingenieria.emiss19H_de_lannoy) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)

In [43]:
X = tds_base.drop_vars(["fwns"]).to_array(dim="channel")

X = X.stack(
    feature=("channel", "polarization", "frequency_band")
)

y = tds_base["fwns"]

In [44]:
X_valid = X.isel(
    lat=slice(RADIUS, -RADIUS),
    lon=slice(RADIUS, -RADIUS)
)

y_valid = y.isel(
    lat=slice(RADIUS, -RADIUS),
    lon=slice(RADIUS, -RADIUS)
)

In [45]:
X_windows = (
    X.rolling(
        lat=WINDOW_SIZE,
        lon=WINDOW_SIZE,
        center=True
    )
    .construct(
        lat="lat_window",
        lon="lon_window"
    )
)

In [46]:
X_windows = X_windows.sel(
    lat=X_valid.lat,
    lon=X_valid.lon
)

In [47]:
X_nn = X_windows.stack(
    sample=("lat", "lon")
)

y_nn = y_valid.stack(
    sample=("lat", "lon")
)

In [56]:
X_np = X_nn.transpose(
    "sample", "feature", "lat_window", "lon_window"
).values
y_np = y_nn.values

In [57]:
mean = X_np.mean(axis=(0, 2, 3), keepdims=True)
std  = X_np.std(axis=(0, 2, 3), keepdims=True) + 1e-6

X_np = (X_np - mean) / std

In [58]:
X_np.shape


(1028176, 44, 5, 5)

In [59]:
y_np.shape

(1028176,)